# Medical RAG Agent Quick Start

This notebook is the Kaggle-friendly single-T4 path for setting up the repo, syncing dependencies with `uv`, running the full judged evaluation, comparing the fine-tuned adapter, and generating figures that match `experiments/all_results.csv`.

If you need access to a private GitHub repo or a gated Hugging Face model, enter the tokens when prompted.

In [ ]:
from pathlib import Path
import os
import shutil
import subprocess
import sys

REPO_NAME = "medical-rag-agent"
REPO_URL = "https://github.com/tsechinchi/medical-rag-agent"


def find_repo_root(start: Path) -> Path | None:
    for candidate in [start, *start.parents]:
        if (candidate / "pyproject.toml").exists() and (candidate / "notebooks").exists():
            return candidate
    return None


start = Path.cwd().resolve()
repo_root = find_repo_root(start)
if repo_root is None:
    repo_root = start / REPO_NAME
    if not repo_root.exists():
        subprocess.run(["git", "clone", REPO_URL, str(repo_root)], check=True)

repo_root = repo_root.resolve()
os.chdir(repo_root)
REPO = str(repo_root)
print(f"Running from: {REPO}")

subprocess.run([sys.executable, "-m", "pip", "install", "-q", "uv"], check=True)
UV_EXE = shutil.which("uv") or shutil.which("uv.exe")
if UV_EXE is None:
    raise RuntimeError("uv was not found on PATH after installation.")

subprocess.run([UV_EXE, "python", "pin", "3.11"], cwd=REPO, check=True)
subprocess.run([UV_EXE, "sync", "--extra", "gpu"], cwd=REPO, check=True)

In [ ]:
# Optional: refresh an existing clone before running the evaluation flow.
# Skip this if you want reproducible results from the current checkout.
# import subprocess
# subprocess.run(["git", "pull", "--ff-only"], cwd=REPO, check=True)

In [ ]:
import subprocess


def run_cmd(args: list[str]) -> subprocess.CompletedProcess[str]:
    result = subprocess.run(args, cwd=REPO, capture_output=True, text=True)
    tail_out = result.stdout[-4000:] if result.stdout else ""
    tail_err = result.stderr[-1000:] if result.stderr else ""
    if tail_out:
        print(tail_out)
    if tail_err:
        print(tail_err)
    if result.returncode != 0:
        raise RuntimeError(f"Command failed with exit code {result.returncode}: {' '.join(args)}")
    return result


print(f"Active repo: {REPO}")

In [ ]:
# Export keys
# In Kaggle, set this as a notebook secret or environment variable before running the login/eval cells.
%env HF_TOKEN=


In [ ]:
from huggingface_hub import login
import os

token = os.getenv("HF_TOKEN", "").strip()

if token:
    login(token=token)
else:
    print("Skipping Hugging Face login because HF_TOKEN is not set.")

In [ ]:
# Optional fast smoke test (10 questions, no BERTScore)
# run_cmd([
#     "uv", "run", "python", "-m", "src.evaluation.run_eval",
#     "--n_samples", "10",
#     "--profile", "fast",
#     "--skip-bertscore",
# ])

In [ ]:
# Full evaluation for the single-T4 budgeted path
run_cmd([
    "uv", "run", "python", "-m", "src.evaluation.run_eval",
    "--profile", "t4-safe",
    "--budget-seconds", "10800",
    "--with-ragas-judge",
])

In [ ]:
# Fine-tuned comparison run (ablation D only)
run_cmd([
    "uv", "run", "python", "-m", "src.evaluation.ablations",
    "--only", "D",
    "--profile", "t4-tight",
    "--budget-seconds", "10800",
    "--with-ragas-judge",
])

In [ ]:
# Optional extra ablations for A/B/C on the fast path
for label in ["A", "B", "C"]:
    run_cmd([
        "uv", "run", "python", "-m", "src.evaluation.ablations",
        "--only", label,
        "--profile", "fast",
    ])

In [ ]:
# Regenerate the merged summary and figures from the latest raw outputs
run_cmd([
    "uv", "run", "python", "-m", "src.evaluation.plot_results",
])

In [ ]:
from datetime import datetime
from pathlib import Path

print(f"Active repo: {REPO}")
artifacts = [
    "experiments/model_free_eval_results.csv",
    "experiments/ablation_qlora.csv",
    "experiments/all_results.csv",
    "docs/figures/bertscore_distribution.png",
    "docs/figures/loop_iterations.png",
    "docs/figures/precision_at_k.png",
    "docs/figures/ragas_radar.png",
]

for rel_path in artifacts:
    path = Path(REPO) / rel_path
    if path.exists():
        modified = datetime.fromtimestamp(path.stat().st_mtime).isoformat(sep=" ", timespec="seconds")
        print(f"{rel_path}: {modified}")
    else:
        print(f"{rel_path}: missing")